In [1]:
import pandas as pd
import numpy as np

import geopandas as gpd
from geopandas.tools import geocode
import multiprocessing as mp
from tqdm import tqdm   # progress bar for loops
from shapely.geometry import Point, LineString, Polygon, MultiPoint

from pyproj import CRS
import matplotlib.patches as mpatches
from tqdm import tqdm
from matplotlib.lines import Line2D
import json
from shapely.ops import linemerge, unary_union, polygonize
from shapely.geometry import LineString, Polygon

import matplotlib.pyplot as plt 

import warnings
warnings.filterwarnings('ignore') 

# Google Earth Engine (pip install earthengine-api)

import geemap, ee

In [2]:
try:
        ee.Initialize()
        
except Exception as e:
    
        ee.Authenticate()
        ee.Initialize()


Successfully saved authorization token.


# VIIRS (2013-2020)

In [3]:
# Viirs image collection

viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG").filterDate('2013-01-01','2020-12-31').select('avg_rad')

In [4]:
# shapefile to Json for constructing an ee object

def shp_to_ee_fmt(geodf):
    data = json.loads(geodf.to_json())
    return data['features']


In [5]:
# Function to reduce nightlight pixel by geopolitical geometry

def reduce_mean(img, shapefile):
    
    try:
        mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=shapefile, scale=30, maxPixels= 1e9).get('avg_rad') # adjust the scale to 30 m
        output = img.set('date', img.date().format()).set('light_mean', mean)
        
    except:
        
        try:
            
            mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=shapefile, scale=50, maxPixels= 1e9).get('avg_rad') # adjust the scale to 50 m
            output = img.set('date', img.date().format()).set('light_mean', mean)
        
        except:
            
            try:
                
                mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=shapefile, scale=100, maxPixels= 1e9).get('avg_rad') # adjust the scale to 100 m
                output = img.set('date', img.date().format()).set('light_mean', mean)
            
            except:
                
                try:
                    
                    mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=shapefile, scale=500, maxPixels= 1e9).get('avg_rad') # adjust the scale to 500 m
                    output = img.set('date', img.date().format()).set('light_mean', mean)
                    
                except:
                    
                    try: 
                    
                        mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=shapefile, scale=1000, maxPixels= 1e9).get('avg_rad') # adjust the scale to 1000 m
                        output = img.set('date', img.date().format()).set('light_mean', mean)
                    
                    except:
                        
                        mean = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=shapefile, scale=1500, maxPixels= 1e9).get('avg_rad') # adjust the scale to 1500 m
                        output = img.set('date', img.date().format()).set('light_mean', mean)
                        
                    
            
    
    
    return output

    

In [6]:
grids = gpd.read_file("../../1.Data/2.silver_data/INEI/GridsCensusEnaho.shp", 
                          crs = "EPSG:4326")

grids.columns

Index(['ID', 'ubigeo', 'lng', 'lat', 'area', 'd10percent', 'pob_total',
       'age1564', 'dfemale', 'nativelang', 'educsup', 'workers', 'work_agri',
       'work_minin', 'work_comme', 'work_serv', 'pwork_agri', 'pwork_mini',
       'pwork_comm', 'pwork_serv', 'page1564', 'pnativelan', 'peducsup',
       'ppea_ocu', 'pobreza', 'dmissingCe', 'dmissingPo', 'ipcr_0', 'ipcr_1',
       'gpgru0', 'dmissingEn', 'zona', 'humid_fore', 'dry_forest', 'desert',
       'steppe', 'shrubland', 'woodland', 'snow', 'paramo', 'tundra',
       'nld_2013', 'nld_2014', 'nld_2015', 'nld_2016', 'nld_2017', 'nld_2018',
       'nld_2019', 'nld_2020', 'geometry'],
      dtype='object')

In [7]:
# Loop to get night-light and transformation by each geopolitical geometry 


grids = gpd.read_file("../../1.Data/2.silver_data/INEI/GridsCensusEnaho_aux.shp", 
                          crs = "EPSG:4326")

json_grids = shp_to_ee_fmt(grids)    # Geopandas to Json file

total_data = {}

for i in tqdm(range(len(json_grids))):   # Loop to get night-light and transformation by each geopolitical geometry

    geometry = ee.Geometry.MultiPolygon(json_grids[i]['geometry']['coordinates'])
    reduced_cities = viirs.map(lambda x: reduce_mean(x,geometry))
    nested_list = reduced_cities.reduceColumns(ee.Reducer.toList(2), ['date','light_mean']).values().get(0)
    df = pd.DataFrame(nested_list.getInfo(), columns=['date','light_mean'])
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = pd.DatetimeIndex(df['date']).year
    
    df['ID'] = json_grids[i]['properties']["ID"]
    df['ubigeo'] = json_grids[i]['properties']["ubigeo"]
    
    total_data[f"data_{i}"] = df


    

100%|██████████| 19580/19580 [3:46:25<00:00,  1.44it/s]  


In [8]:
df = pd.concat(total_data.values(), ignore_index=True)

In [9]:

df.to_stata(fr"..\..\1.Data\2.silver_data\RemoteSensingData\VIIRS\NLD_Grids1km_2013_2020.dta",
                                write_index = False)
        

## District Level 2013

In [10]:
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG").filterDate('2016-01-01','2016-12-31').select('avg_rad')

In [11]:
grids = gpd.read_file("../../1.Data/1.raw_data/shapefile/Limites_administrativos_2007/Distritos.shp", 
                          crs = "EPSG:4326")
grids.columns

Index(['CODIGO_DEP', 'NOMBDPTO', 'CODIGO_PRO', 'NOMBPROV', 'CODIGO_DIS',
       'NOMBDIST', 'CAPITAL', 'geometry'],
      dtype='object')

In [12]:
grids['ubigeo'] = grids['CODIGO_DEP'] + grids['CODIGO_PRO'] + grids['CODIGO_DIS']
grids

,CODIGO_DEP,NOMBDPTO,CODIGO_PRO,NOMBPROV,CODIGO_DIS,NOMBDIST,CAPITAL,geometry,ubigeo
0,23,TACNA,01,TACNA,10,CORONEL GREGORIO ALBARRACIN LANCHIPA,ALFONSO UGARTE,"POLYGON ((-70.14409 -18.09309, -70.14416 -18.0...",230110
1,23,TACNA,01,TACNA,08,POCOLLAY,POCOLLAY,"POLYGON ((-69.90467 -17.95829, -69.90608 -17.9...",230108
2,23,TACNA,01,TACNA,03,CALANA,CALANA,"POLYGON ((-70.09201 -17.98026, -70.09409 -17.9...",230103
3,23,TACNA,01,TACNA,01,TACNA,TACNA,"POLYGON ((-70.23500 -17.99231, -70.23399 -17.9...",230101
4,23,TACNA,01,TACNA,09,SAMA,LAS YARAS,"POLYGON ((-70.42425 -17.88957, -70.42374 -17.8...",230109
...,...,...,...,...,...,...,...,...,...
1827,16,LORETO,01,MAYNAS,14,TENIENTE MANUEL CLAVERO,SOPLIN VARGAS,"POLYGON ((-73.53075 -1.45793, -73.54104 -1.458...",160114
1828,06,CAJAMARCA,09,SAN IGNACIO,06,SAN JOSE DE LOURDES,SAN JOSE DE LOURDES,"POLYGON ((-78.76924 -4.64055, -78.76907 -4.640...",060906
1829,06,CAJAMARCA,08,JAEN,12,SANTA ROSA,SANTA ROSA,"POLYGON ((-78.67973 -5.40859, -78.67897 -5.408...",060812
1830,01,AMAZONAS,04,CONDORCANQUI,02,EL CENEPA,HUAMPAMI,"POLYGON ((-78.21812 -3.50889, -78.21822 -3.509...",010402


In [13]:
# Loop to get night-light and transformation by each geopolitical geometry 

# grids = gpd.read_file("../../1.Data/1.raw_data/shapefile/Limites_administrativos_2007/Distritos.shp", 
#                           crs = "EPSG:4326")

json_grids = shp_to_ee_fmt(grids)    # Geopandas to Json file

total_data = {}

for i in tqdm(range(len(json_grids))):   # Loop to get night-light and transformation by each geopolitical geometry

    geometry = ee.Geometry.MultiPolygon(json_grids[i]['geometry']['coordinates'])
    reduced_cities = viirs.map(lambda x: reduce_mean(x,geometry))
    nested_list = reduced_cities.reduceColumns(ee.Reducer.toList(2), ['date','light_mean']).values().get(0)
    df = pd.DataFrame(nested_list.getInfo(), columns=['date','light_mean'])
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = pd.DatetimeIndex(df['date']).year
    df['ubigeo'] = json_grids[i]['properties']["ubigeo"]
    
    total_data[f"data_{i}"] = df


    

100%|██████████| 1832/1832 [17:12<00:00,  1.77it/s]


In [14]:
df = pd.concat(total_data.values(), 
               ignore_index=True)

In [15]:

df.to_stata(fr"..\..\1.Data\2.silver_data\RemoteSensingData\VIIRS\NLD_district_2016.dta",
                                write_index = False)
        